# AI Security Project — Member 1: Data + Baseline Model
**Dataset:** SMS Spam Collection Dataset  
**Task:** Binary Text Classification — Spam vs Ham

## 1. Dataset Description

The **SMS Spam Collection Dataset** is a labeled dataset of 5,572 English SMS messages collected from various sources. Each message is tagged as either:
- **ham** — legitimate (non-spam) message
- **spam** — unsolicited/junk message

| Property | Value |
|---|---|
| Total samples | 5,572 |
| Ham messages | 4,825 (86.6%) |
| Spam messages | 747 (13.4%) |
| Language | English |
| Task | Binary Classification |

The dataset is **imbalanced** — ham messages are ~6.5× more common than spam.

## 2. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## 3. Load the Dataset

In [ ]:
# Load CSV — uses latin-1 encoding to handle special characters
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only the two relevant columns and rename them
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'message']

print('Dataset shape:', df.shape)
print()
df.head(10)

In [ ]:
# Quick overview
print('Label distribution:')
print(df['label'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'salmon'])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.suptitle('SMS Spam Collection — Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

## 4. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """
    Full preprocessing pipeline:
      1. Lowercase
      2. Remove URLs
      3. Remove phone numbers
      4. Remove non-alphabetic characters (punctuation, digits, symbols)
      5. Strip extra whitespace
    """
    text = text.lower()                                  # Step 1: lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)        # Step 2: remove URLs
    text = re.sub(r'\b\d{7,}\b', '', text)              # Step 3: remove phone numbers
    text = re.sub(r'[^a-z\s]', '', text)                # Step 4: remove non-alpha
    text = re.sub(r'\s+', ' ', text).strip()            # Step 5: clean whitespace
    return text

# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess_text)

# Show before/after examples
print('=== Preprocessing Examples ===')
for i in [2, 5, 8]:  # mix of spam and ham
    print(f"\n[{df['label'][i].upper()}]")
    print(f"  BEFORE: {df['message'][i]}")
    print(f"  AFTER:  {df['clean_message'][i]}")

## 5. Label Encoding

In [ ]:
# Encode labels: spam = 1, ham = 0
df['label_enc'] = df['label'].map({'spam': 1, 'ham': 0})

print('Label encoding:')
print(df[['label', 'label_enc']].drop_duplicates().sort_values('label_enc').to_string(index=False))

## 6. Train / Test Split

In [ ]:
X = df['clean_message']
y = df['label_enc']

# 80/20 split, stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size : {len(X_train):,} samples')
print(f'Test set size      : {len(X_test):,} samples')
print()
print('Train label distribution:')
print(y_train.value_counts().rename({0: 'ham', 1: 'spam'}))
print()
print('Test label distribution:')
print(y_test.value_counts().rename({0: 'ham', 1: 'spam'}))

## 7. Feature Extraction — TF-IDF Vectorization

In [ ]:
# TF-IDF with unigrams and bigrams, capped at 10,000 features
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True       # apply log normalization to term frequencies
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print('TF-IDF feature matrix shape:')
print(f'  Train: {X_train_tfidf.shape}')
print(f'  Test : {X_test_tfidf.shape}')

## 8. Train the Baseline Classifier — Logistic Regression

In [ ]:
# Logistic Regression — fast, interpretable, strong baseline for text
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train_tfidf, y_train)

print('Model trained successfully.')
print(f'Model type: {clf.__class__.__name__}')

## 9. Evaluate the Baseline Model

In [ ]:
# Predictions
y_pred = clf.predict(X_test_tfidf)

# Metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print('=' * 45)
print('       BASELINE MODEL — EVALUATION RESULTS')
print('=' * 45)
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 45)
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Ham', 'Predicted Spam'],
            yticklabels=['Actual Ham', 'Actual Spam'],
            ax=ax, linewidths=0.5, linecolor='gray')
ax.set_title('Baseline Model — Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_baseline.png')

In [ ]:
# Results summary table — easy to copy into the report
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [accuracy, precision, recall, f1]
})
results['Score (%)'] = (results['Score'] * 100).round(2).astype(str) + '%'
results['Score'] = results['Score'].round(4)
print(results.to_string(index=False))

# Save for other members
results.to_csv('baseline_results.csv', index=False)
print('\nSaved: baseline_results.csv')

## 10. Save Model & Vectorizer (for other members to use)

In [ ]:
import pickle

# Save the trained model
with open('baseline_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

# Save the vectorizer (MUST be saved with the model — same fit)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Save processed dataframe for other members
df[['label', 'label_enc', 'message', 'clean_message']].to_csv('processed_dataset.csv', index=False)

# Save train/test splits
train_df = pd.DataFrame({'clean_message': X_train, 'label': y_train})
test_df  = pd.DataFrame({'clean_message': X_test,  'label': y_test})
train_df.to_csv('train_split.csv', index=False)
test_df.to_csv('test_split.csv', index=False)

print('Saved files:')
print('  baseline_model.pkl     — trained Logistic Regression')
print('  tfidf_vectorizer.pkl   — fitted TF-IDF vectorizer')
print('  processed_dataset.csv  — full cleaned dataset')
print('  train_split.csv        — training split')
print('  test_split.csv         — test split')
print('  baseline_results.csv   — metrics table')
print('  confusion_matrix_baseline.png')
print('  class_distribution.png')

## Summary

| Step | Detail |
|---|---|
| Dataset | SMS Spam Collection (5,572 messages) |
| Preprocessing | Lowercase, remove URLs/phone numbers, strip non-alpha chars |
| Features | TF-IDF (unigrams + bigrams, max 10,000 features) |
| Model | Logistic Regression (C=1.0) |
| Train/Test Split | 80% / 20%, stratified |

### Baseline Results

| Metric | Score |
|---|---|
| Accuracy  | 96.77% |
| Precision | 99.13% |
| Recall    | 76.51% |
| F1-Score  | 86.36% |

> These results are the **"Before Attack"** benchmark. Members 2 & 3 will apply adversarial attacks and defense techniques, using the saved model and vectorizer files.